# 03 - Quality gate (Great Expectations) and lineage (OpenLineage)
Rubric deliverable 5.

In [1]:
import os
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) == 'notebooks' else None
os.environ.setdefault('TQDM_DISABLE', '1')

'1'

## 1. The gate passes on the real Silver table

In [2]:
from src.quality.expectations import run_silver_quality_gate, QualityGateError
res = run_silver_quality_gate()
print('success:', res.success, '|', res.statistics['successful_expectations'],
      '/', res.statistics['evaluated_expectations'], 'expectations met')

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/4 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/52 [00:00<?, ?it/s]

success: True | 14 / 14 expectations met


## 2. The gate fails - and this is what halts the pipeline before Gold

In [3]:
from src.lakehouse.silver import read_silver
df = read_silver().to_pandas()
df.loc[df.index[0], 'heart_rate'] = 900
df.loc[df.index[1], 'spo2'] = 20
df.loc[df.index[2], 'reading_id'] = df.loc[df.index[3], 'reading_id']
try:
    run_silver_quality_gate(df)
except QualityGateError as e:
    print(e)

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/4 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/52 [00:00<?, ?it/s]

Silver quality gate failed: 3 expectation(s) not met
  - expect_column_values_to_be_unique(reading_id): 2 unexpected
  - expect_column_values_to_be_between(heart_rate): 1 unexpected
  - expect_column_values_to_be_between(spo2): 1 unexpected


## 3. OpenLineage START / COMPLETE / FAIL per stage
`Lineage.stage()` emits START on entry, COMPLETE on success, FAIL (with an error facet) on exception.

In [4]:
import json, pathlib
from src.lineage.emit import Lineage
p = pathlib.Path('lineage/nb_demo.jsonl'); p.unlink(missing_ok=True)
lin = Lineage(run_id='00000000-0000-0000-0000-0000000000nb'.replace('nb','ab'), event_file_path=p)
with lin.stage('build_silver', inputs=['delta.bronze'], outputs=['delta.silver']):
    pass
try:
    with lin.stage('quality_gate', inputs=['delta.silver']):
        raise RuntimeError('gate failed on batch')
except RuntimeError:
    pass
for line in p.read_text().splitlines():
    e = json.loads(line)
    extra = ' | ' + e['run']['facets']['errorMessage']['message'] if e['eventType']=='FAIL' else ''
    print(f"{e['eventType']:9} {e['job']['name']}  parent={e['run']['facets']['parent']['run']['runId']}{extra}")

START     build_silver  parent=00000000-0000-0000-0000-0000000000ab
COMPLETE  build_silver  parent=00000000-0000-0000-0000-0000000000ab
START     quality_gate  parent=00000000-0000-0000-0000-0000000000ab
FAIL      quality_gate  parent=00000000-0000-0000-0000-0000000000ab | RuntimeError: gate failed on batch
